# 05 — Evidence gate, abstention et conflits

**Objectif** : démontrer que `COMPLETE`, `PARTIAL`, `NOT_FOUND` et `CONFLICT` sont décidés par du code et non par la confiance du modèle.

**Critère de passage** : chaque statut est reproductible et chaque assertion est rattachée à une preuve acceptée.

In [ ]:
from pathlib import Path
import sys

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table

ROOT = bootstrap()
from app.domain.models import AnswerStatus
from app.domain.profiles import map_question
from app.evidence.gate import DeterministicEvidenceGate
from app.retrieval.hybrid import HybridRetriever
from app.retrieval.planner import plan_queries
from app.store.artifacts import store

In [ ]:
question = 'Quels indicateurs publics décrivent la position du Groupe en 2025 ?'
document_ids = ['foyer_financial_information_2025']
profile = map_question(question)
queries = plan_queries(question, profile, document_ids)
retriever = HybridRetriever(store.selected_chunks(document_ids))
candidates_by_field = [retriever.search(query, k=5) for query in queries]
all_candidates = [candidate for batch in candidates_by_field for candidate in batch]
gate = DeterministicEvidenceGate()
complete_result = gate.evaluate(profile, all_candidates)
partial_result = gate.evaluate(profile, candidates_by_field[0])
assert complete_result.status == AnswerStatus.COMPLETE
assert partial_result.status == AnswerStatus.PARTIAL
{
    'complete': complete_result.status.value,
    'partial': partial_result.status.value,
    'partial_missing_fields': partial_result.missing_fields,
}

In [ ]:
empty_result = gate.evaluate(profile, [])
assert empty_result.status == AnswerStatus.NOT_FOUND
display_table([
    {'champ': coverage.field_id, 'statut': coverage.state}
    for coverage in empty_result.coverage
])

In [ ]:
equity_candidate = next(
    candidate for candidate in all_candidates
    if candidate.field_id == 'group_equity'
    and any(fact.field_id == 'group_equity' for fact in candidate.chunk.facts)
)
original_fact = next(fact for fact in equity_candidate.chunk.facts if fact.field_id == 'group_equity')
conflicting_fact = original_fact.model_copy(update={'value': 999.0, 'formatted_value': '999,0 M€'})
conflicting_chunk = equity_candidate.chunk.model_copy(update={
    'id': f'{equity_candidate.chunk.id}-synthetic-conflict',
    'facts': [conflicting_fact],
})
conflicting_candidate = equity_candidate.model_copy(update={'chunk': conflicting_chunk})
conflict_result = gate.evaluate(profile, [*all_candidates, conflicting_candidate])
assert conflict_result.status == AnswerStatus.CONFLICT
display_table([
    {'champ': item.field_id, 'état': item.state, 'valeur': item.fact.formatted_value if item.fact else None}
    for item in conflict_result.evidence if item.field_id == 'group_equity'
])

In [ ]:
accepted = [item for item in complete_result.evidence if item.state == 'ACCEPTED']
assert all(item.source and item.source.document_id for item in accepted)
assert all(item.fact and item.fact.entity for item in accepted)
assert all(item.excerpt for item in accepted)
{
    'accepted_evidence': len(accepted),
    'all_have_source': True,
    'all_have_entity': True,
    'all_have_excerpt': True,
}